# 01 - Data Ingestion and Randomization Check

**Purpose:** Ingest the raw Hillstrom email campaign dataset, isolate the Women's E-Mail treatment arm against the No E-Mail control, persist the experiment dataset to SQLite and raw CSV, and verify treatment allocation with a Sample Ratio Mismatch (SRM) test.

**Inputs:** Hillstrom MineThatData E-Mail Analytics dataset (via `sklift.datasets.fetch_hillstrom`).

**Outputs:** `../data/raw/hillstrom_raw.csv`, SQLite table `raw_customers` and `experiment` in `../data/hillstrom.db`.

In [1]:
# ==========================================
# Data Ingestion
# ==========================================
import pandas as pd
from sklift.datasets import fetch_hillstrom

# Fetch all targets to preserve visit, conversion, and spend
bunch = fetch_hillstrom(target_col='all')

# Reassemble
df = bunch.data.copy()
df['treatment'] = bunch.treatment
df['visit'] = bunch.target['visit']
df['conversion'] = bunch.target['conversion']
df['spend'] = bunch.target['spend']

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nTreatment allocation:")
print(df['treatment'].value_counts())

print("\nVisit distribution:")
print(df['visit'].value_counts())

print("\nConversion distribution:")
print(df['conversion'].value_counts())

print("\nSpend summary (guardrail):")
print(df['spend'].describe().round(2))

Dataset shape: (64000, 12)

First 5 rows:
   recency history_segment  history  mens  womens   zip_code  newbie channel  \
0       10  2) $100 - $200   142.44     1       0  Surburban       0   Phone   
1        6  3) $200 - $350   329.08     1       1      Rural       1     Web   
2        7  2) $100 - $200   180.65     0       1  Surburban       1     Web   
3        9  5) $500 - $750   675.83     1       0      Rural       1     Web   
4        2    1) $0 - $100    45.34     1       0      Urban       0     Web   

       treatment  visit  conversion  spend  
0  Womens E-Mail      0           0    0.0  
1      No E-Mail      0           0    0.0  
2  Womens E-Mail      0           0    0.0  
3    Mens E-Mail      0           0    0.0  
4  Womens E-Mail      0           0    0.0  

Treatment allocation:
treatment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

Visit distribution:
visit
0    54606
1     9394
Name: count, dtype: int64

Con

In [2]:
# ==========================================
# SQLite Storage & Round-Trip Check
# ==========================================
import sqlite3

conn = sqlite3.connect('../data/hillstrom.db')
df.to_sql('raw_customers', conn, if_exists='replace', index=False)

# Round-trip check: verify SQLite table row count and sample against in-memory DataFrame
print("SQLite row count:")
print(pd.read_sql('SELECT COUNT(*) AS row_count FROM raw_customers', conn))

print("\nSQLite round-trip sample (LIMIT 5):")
print(pd.read_sql('SELECT * FROM raw_customers LIMIT 5', conn))

SQLite row count:
   row_count
0      64000

SQLite round-trip sample (LIMIT 5):
   recency history_segment  history  mens  womens   zip_code  newbie channel  \
0       10  2) $100 - $200   142.44     1       0  Surburban       0   Phone   
1        6  3) $200 - $350   329.08     1       1      Rural       1     Web   
2        7  2) $100 - $200   180.65     0       1  Surburban       1     Web   
3        9  5) $500 - $750   675.83     1       0      Rural       1     Web   
4        2    1) $0 - $100    45.34     1       0      Urban       0     Web   

       treatment  visit  conversion  spend  
0  Womens E-Mail      0           0    0.0  
1      No E-Mail      0           0    0.0  
2  Womens E-Mail      0           0    0.0  
3    Mens E-Mail      0           0    0.0  
4  Womens E-Mail      0           0    0.0  


### Design Decision: Two-Arm Focus (Dropping Men's E-Mail)

The Hillstrom dataset includes two distinct marketing campaigns alongside a holdout control: *Mens E-Mail* (~21,300), *Womens E-Mail* (~21,400), and *No E-Mail* (~21,300).

We drop the Men's arm and restrict our analysis to **Women's E-Mail vs. No E-Mail**. This frames a clean binary treatment evaluation ($T \in \{0, 1\}$), isolating the causal effect of a single creative and product intervention without diluting treatment sample size across competing offers or confounding two distinct campaign strategies.

In [3]:
# ==========================================
# Two-Arm Experiment Preparation
# ==========================================
import os

os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/hillstrom_raw.csv', index=False)

conn.execute("DROP TABLE IF EXISTS experiment")
conn.execute("""
    CREATE TABLE experiment AS
    SELECT * FROM raw_customers
    WHERE treatment != 'Mens E-Mail';
""")
conn.commit()

df_exp = pd.read_sql("SELECT * FROM experiment", conn)
print(f"Row count after dropping Mens E-Mail: {len(df_exp)}")

pd.read_sql(
    "SELECT treatment, COUNT(*) AS n FROM experiment GROUP BY treatment",
    conn
)

Row count after dropping Mens E-Mail: 42693


,treatment,n
0,No E-Mail,21306
1,Womens E-Mail,21387


In [4]:
# ==========================================
# Sample Ratio Mismatch (SRM) Test
# ==========================================
from scipy.stats import chisquare

counts = df_exp['treatment'].value_counts()

stat, pval = chisquare(counts, f_exp=[counts.sum()/2, counts.sum()/2])
print(f"Chi-square p-value: {pval:.4f}")

Chi-square p-value: 0.6950


The SRM chi-square test produced a p-value of 0.6950, so we did not find evidence of a mismatch from the expected 50/50 treatment allocation.